# H-006 · Does Proximity to the 52-Week High Predict Forward Returns?

Factor test for **H-006** (equities): whether stocks closer to their rolling peak high earn higher next-week returns than names in deeper drawdowns (George & Hwang 2004 anchoring / underreaction).

- **Idea** — Ratio of today’s close to the highest high over the prior `W` trading days (paper default **252** ≈ 52 weeks). Near 1.0 = trading near its high; near 0 = deep drawdown from the high.
- **Claim** — Stocks closer to their 52-week high outperform stocks farther from it over the next 5–21 days (positive IC).
- **Why it might work** — Investors use the 52-week high as a reference point and underreact to good news for names near the high. This is a *level-of-path* signal, not a return-over-lookback signal like H-001 or H-004 residual momentum.
- **Data** — Daily OHLCV long panel (`close`, `high`) from `s1_factor_panel_train.parquet`.

**No floor / no winsorize in the feature store:** `add_near_52w_factors` does **not** floor the peak denominator (non-positive `Hmax` → NaN) and does **not** winsorize. If you need winsorization, apply it in §2 — not inside the factor API.

**This notebook screens** `WINDOWS × MODES × NORMALIZE_OPTS` on research IS only (16 columns). Store names do not encode `normalize`, so columns are renamed to `near_52w_{mode}_{W}_{raw|cs}` for the screen.

| Knob | Values |
|------|--------|
| `WINDOWS` | `[63, 126, 252, 504]` (≈3m / 6m / **1y paper default** / 2y) |
| `MODES` | `ratio`, `log_drawdown` |
| `NORMALIZE_OPTS` | `False` → tag `raw`; `True` → tag `cs` (CS pct-rank by date) |
| Alphalens `periods` | `(1, 5, 21)` (primary narrative **5d**) |

**`mode` options:**

| Mode | Behavior |
|------|----------|
| `ratio` (default) | `close / Hmax` (high = nearer the peak). |
| `log_drawdown` | `ln(close / Hmax)` when ratio `> 0`; non-positive → NaN. |

Today is **included** in the rolling peak. Do **not** re-split the train parquet; do **not** use `s1_factor_panel_full.parquet` for window keep/kill.

**No baseline factors** in this notebook (no raw mom / H-001 / H-004). Compare survivors to momentum-family factors in a later pass if needed.

Use `data.processing.s1_feature_store.add_near_52w_factors` rather than reimplementing the factor inline.

Evaluation uses the S1 **trade-date** panel: Alphalens pivots `open` (no `shift(-1)`); labels are open-to-open. Prior close-to-close ICs are not comparable.


## 0. Imports & Config

Resolve the repo root; configure `WINDOWS`, `MODES`, `NORMALIZE_OPTS`, Alphalens `PERIODS`, and tearsheet paths. The train parquet is already research IS; do not calculate another cutoff here.

In [1]:
import os
import sys

import alphalens as al
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd

from data.processing.cleaner import forward_fill_panel
from data.processing.s1_feature_store import add_near_52w_factors

# Jupyter cwd is often this notebook's folder, not the repo root; walk up until we find 01_data/ingestion.
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

TRAIN_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)

TEARSHEET_DIR = os.path.join(
    ROOT, "02_research", "notebooks", "s1_equities", "factor_tests", "tearsheets"
)

# --- Window screen (trading days; 252 = paper / George–Hwang default) ---
WINDOWS = [63, 126, 252, 504]

# --- feature_subset IDs screened ---
MODES = ["ratio", "log_drawdown"]

# --- Normalize screen (False → raw unitless ratio; True → CS pct-rank) ---
NORMALIZE_OPTS = [False, True]

# --- Fixed for this notebook ---
PERIODS = (1, 5, 21)  # S1 default; primary narrative = 5d
QUANTILES = 5
MAX_LOSS = 0.35


## 1. Data Loading

Load `s1_factor_panel_train.parquet`, which contains the daily OHLCV chronological research IS selected by `RESEARCH_IS_FRACTION` in the panel notebook. Confirm columns: `date`, `ticker`, `close`, `high`.

Do not apply another 70/30 split in this notebook. Do not use `s1_factor_panel_full.parquet` for window keep/kill.

In [2]:
panel = pd.read_parquet(TRAIN_PANEL_PATH)
required = {"date", "ticker", "open", "high", "close", "feature_date"}
missing = required - set(panel.columns)
if missing:
    raise ValueError(f"train panel missing columns: {sorted(missing)}")

panel = panel.copy()
panel["date"] = pd.to_datetime(panel["date"])
print(
    f"rows={len(panel):,}  tickers={panel['ticker'].nunique():,}  "
    f"dates={panel['date'].nunique():,}  "
    f"[{panel['date'].min().date()} → {panel['date'].max().date()}]"
)
panel.head()


rows=289,381  tickers=100  dates=2,915  [2010-01-05 → 2021-08-03]


,date,ticker,open,high,low,close,volume,feature_date,fwd_ret_1,fwd_ret_5,fwd_ret_21
0,2010-01-05,AAPL,6.424143,6.421146,6.357683,6.406478,493729600.0,2010-01-04,-0.001025,-0.025210,-0.083271
1,2010-01-06,AAPL,6.417558,6.453779,6.383729,6.417557,601904800.0,2010-01-05,-0.012268,-0.030367,-0.101456
2,2010-01-07,AAPL,6.338825,6.443003,6.308892,6.315478,552160000.0,2010-01-06,-0.006848,-0.007745,-0.075844
3,2010-01-08,AAPL,6.295419,6.346309,6.258000,6.303801,477131200.0,2010-01-07,0.011888,0.002996,-0.066001
4,2010-01-11,AAPL,6.370258,6.346310,6.258300,6.345711,447610800.0,2010-01-08,-0.016965,-0.021005,-0.079464


## 2. Data Cleaning & Engineering

Forward-fill `close` / `high` via the project cleaner, then drop remaining nulls. The store itself applies **no** floor and **no** winsorize.

Point-in-time only: features at `t` use information available at or before `t`.

In [3]:
ohlc = ["close", "high"]
panel = forward_fill_panel(panel, columns=ohlc, limit=5)
panel = panel.dropna(subset=ohlc).reset_index(drop=True)
print(
    f"after clean: rows={len(panel):,}  "
    f"null close/high={panel[ohlc].isna().any(axis=1).sum()}"
)


after clean: rows=289,381  null close/high=0


## 3. Modeling / Signal Construction

Call `add_near_52w_factors` for each `(mode, normalize)` via `feature_subset=[mode]` with `window=WINDOWS` (multi → `near_52w_{mode}_{W}`). Rename into the panel as `near_52w_{mode}_{W}_{raw|cs}` so both normalize settings can coexist.

Do not reimplement the rolling peak / ratio inline.


In [4]:
FACTOR_COLS: list[str] = []

for mode in MODES:
    for normalize in NORMALIZE_OPTS:
        tmp = add_near_52w_factors(
            panel,
            window=WINDOWS,
            feature_subset=[mode],
            normalize=normalize,
        )
        tag = "cs" if normalize else "raw"
        for w in WINDOWS:
            src = f"near_52w_{mode}_{w}"
            dst = f"near_52w_{mode}_{w}_{tag}"
            if src not in tmp.columns:
                raise ValueError(f"expected store column {src!r}, got {list(tmp.columns)}")
            panel[dst] = tmp[src]
            FACTOR_COLS.append(dst)

print(f"near_52w factors ({len(FACTOR_COLS)}):")
for c in FACTOR_COLS:
    print(f"  {c}")


near_52w factors (16):
  near_52w_ratio_63_raw
  near_52w_ratio_126_raw
  near_52w_ratio_252_raw
  near_52w_ratio_504_raw
  near_52w_ratio_63_cs
  near_52w_ratio_126_cs
  near_52w_ratio_252_cs
  near_52w_ratio_504_cs
  near_52w_log_drawdown_63_raw
  near_52w_log_drawdown_126_raw
  near_52w_log_drawdown_252_raw
  near_52w_log_drawdown_504_raw
  near_52w_log_drawdown_63_cs
  near_52w_log_drawdown_126_cs
  near_52w_log_drawdown_252_cs
  near_52w_log_drawdown_504_cs


## 4. Evaluation

Alphalens IC + quintile spreads on research IS only at `periods=(1, 5, 21)` (primary narrative **5d**). Screen all 16 `window × mode × normalize` columns. No baseline factors.

### 4.1 Window × mode × normalize screen

| Token | Meaning |
|-------|---------|
| **mode** | `ratio` or `log_drawdown` |
| **W** | Trading-day peak window (`WINDOWS`) |
| **normalize** | `raw` = store `normalize=False` (default); `cs` = store `normalize=True` (optional CS pct-rank) |

**Column pattern:** `near_52w_{mode}_{W}_{raw|cs}` — e.g. `near_52w_ratio_252_raw`

Primary ranking metric: **mean IC at 5d** (`ic_5d`). After the full table, a pivot compares `raw` vs `cs` for each `(mode, W)`.

In [5]:
def to_alphalens_prices(panel: pd.DataFrame) -> pd.DataFrame:
    """Wide open matrix for Alphalens (trade-date panel; entry at open)."""
    prices = panel.pivot(index="date", columns="ticker", values="open")
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


def to_alphalens_factor(panel: pd.DataFrame, col: str) -> pd.Series:
    """MultiIndex (date, ticker) factor series for Alphalens."""
    factor = panel.set_index(["date", "ticker"])[col].dropna()
    factor.index = factor.index.set_levels(
        pd.to_datetime(factor.index.levels[0]), level=0
    )
    return factor.sort_index()


def parse_factor_name(col: str) -> dict:
    """Decode ``near_52w_{mode}_{W}_{raw|cs}`` (mode may contain underscores)."""
    if not col.startswith("near_52w_"):
        raise ValueError(f"unrecognized factor column: {col!r}")
    rest = col[len("near_52w_"):]
    parts = rest.rsplit("_", 2)
    if len(parts) != 3:
        raise ValueError(
            f"expected near_52w_{{mode}}_{{W}}_{{raw|cs}}, got {col!r}"
        )
    mode, w_str, tag = parts
    if tag not in {"raw", "cs"}:
        raise ValueError(f"expected normalize tag raw|cs, got {tag!r} in {col!r}")
    return {
        "mode": mode,
        "W": int(w_str),
        "normalize": tag == "cs",
        "norm_tag": tag,
    }


def _period_label(period_index: pd.Index, period: int, position: int):
    """Match Alphalens period label ('1D', '5D', …) or fall back by position."""
    for c in (f"{period}D", f"{period}d", period, str(period)):
        if c in period_index:
            return c
    return period_index[position]


def factor_screen_metrics(
    factor: pd.Series,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
) -> dict:
    """Mean IC and Q5−Q1 mean return spread for each forward period."""
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=factor,
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )
    mean_ic = al.performance.mean_information_coefficient(factor_data)
    mean_ret, _ = al.performance.mean_return_by_quantile(factor_data, demeaned=True)

    row = {}
    for i, p in enumerate(periods):
        ic_key = _period_label(mean_ic.index, p, i)
        ret_key = _period_label(mean_ret.columns, p, i)
        row[f"ic_{p}d"] = float(mean_ic.loc[ic_key])
        q_hi, q_lo = mean_ret.index.max(), mean_ret.index.min()
        row[f"spread_{p}d"] = float(
            mean_ret.loc[q_hi, ret_key] - mean_ret.loc[q_lo, ret_key]
        )
    return row


In [6]:
prices = to_alphalens_prices(panel)

rows = []
for col in FACTOR_COLS:
    meta = parse_factor_name(col)
    metrics = factor_screen_metrics(to_alphalens_factor(panel, col), prices)
    rows.append({"factor": col, **meta, **metrics})

summary = (
    pd.DataFrame(rows)
    .sort_values("ic_5d", ascending=False)
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)
summary


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

,factor,mode,W,normalize,norm_tag,ic_1d,spread_1d,ic_5d,spread_5d,ic_21d,spread_21d
0,near_52w_ratio_252_raw,ratio,252,False,raw,0.0134,-0.0000,0.0143,0.0002,0.0207,0.0009
1,near_52w_ratio_252_cs,ratio,252,True,cs,0.0134,-0.0000,0.0143,0.0002,0.0207,0.0009
2,near_52w_log_drawdown_252_cs,log_drawdown,252,True,cs,0.0134,-0.0000,0.0143,0.0002,0.0207,0.0009
3,near_52w_log_drawdown_252_raw,log_drawdown,252,False,raw,0.0134,-0.0000,0.0143,0.0002,0.0207,0.0009
4,near_52w_log_drawdown_126_raw,log_drawdown,126,False,raw,0.0124,-0.0000,0.0130,0.0000,0.0198,0.0002
5,near_52w_log_drawdown_126_cs,log_drawdown,126,True,cs,0.0124,-0.0000,0.0130,0.0000,0.0198,0.0002
6,near_52w_ratio_126_cs,ratio,126,True,cs,0.0124,-0.0000,0.0130,0.0000,0.0198,0.0002
7,near_52w_ratio_126_raw,ratio,126,False,raw,0.0124,-0.0000,0.0130,0.0000,0.0198,0.0002
8,near_52w_log_drawdown_504_cs,log_drawdown,504,True,cs,0.0117,-0.0001,0.0126,-0.0004,0.0152,-0.0009
9,near_52w_log_drawdown_504_raw,log_drawdown,504,False,raw,0.0117,-0.0001,0.0126,-0.0004,0.0152,-0.0009


#### Normalize comparison (`raw` vs `cs`)

Same `(mode, W)` side-by-side: which `normalize` setting has higher `ic_5d`? `delta_ic_5d = cs − raw` (positive ⇒ CS rank more informative at 5d).

In [7]:
norm_cmp = (
    summary.pivot_table(
        index=["mode", "W"],
        columns="norm_tag",
        values="ic_5d",
        aggfunc="first",
    )
    .reindex(columns=["raw", "cs"])
    .sort_index()
)
norm_cmp["delta_ic_5d"] = norm_cmp["cs"] - norm_cmp["raw"]
norm_cmp["winner"] = norm_cmp.apply(
    lambda r: "cs" if r["cs"] > r["raw"] else ("raw" if r["raw"] > r["cs"] else "tie"),
    axis=1,
)
print("normalize winners (by ic_5d):")
print(norm_cmp["winner"].value_counts().to_string())
norm_cmp


normalize winners (by ic_5d):
winner
tie    8


norm_tag            raw     cs  delta_ic_5d winner
mode         W                                    
log_drawdown 63  0.0094 0.0094       0.0000    tie
             126 0.0130 0.0130       0.0000    tie
             252 0.0143 0.0143       0.0000    tie
             504 0.0126 0.0126       0.0000    tie
ratio        63  0.0094 0.0094       0.0000    tie
             126 0.0130 0.0130       0.0000    tie
             252 0.0143 0.0143       0.0000    tie
             504 0.0126 0.0126       0.0000    tie

### 4.2 Full tear sheet (manual pick)

Review §4.1, then set `TEAR_MODE`, `TEAR_WINDOW`, and `TEAR_NORMALIZE` below. Nothing is auto-selected from the winner.

The tear is displayed in-notebook **and** saved as a multi-page PDF under `02_research/notebooks/s1_equities/factor_tests/tearsheets/` named `H-006_{factor_col}.pdf`. Re-running overwrites the same path.

In [8]:
def near_52w_factor_col(mode: str, window: int, normalize: bool) -> str:
    """Column name for a screened near-52w factor."""
    tag = "cs" if normalize else "raw"
    return f"near_52w_{mode}_{window}_{tag}"


def run_full_tear(
    panel: pd.DataFrame,
    factor_col: str,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
    tearsheet_dir: str = TEARSHEET_DIR,
):
    """Build factor_data, run Alphalens full tear, save figs to multi-page PDF.

    Alphalens calls plt.show() after each plot, which clears figures under Agg.
    Temporarily replace plt.show so each figure is written into the PDF before close.
    """
    if factor_col not in panel.columns:
        raise ValueError(
            f"{factor_col!r} not in panel — pick mode/W/normalize that were screened "
            f"(available: {FACTOR_COLS})"
        )
    plt.close("all")
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=to_alphalens_factor(panel, factor_col),
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )

    os.makedirs(tearsheet_dir, exist_ok=True)
    out_path = os.path.join(tearsheet_dir, f"H-006_{factor_col}.pdf")
    pdf = PdfPages(out_path)
    n_pages = 0
    _original_show = plt.show

    def _show_and_savefig(*args, **kwargs):
        nonlocal n_pages
        for num in list(plt.get_fignums()):
            fig = plt.figure(num)
            if fig.axes:
                pdf.savefig(fig, bbox_inches="tight")
                n_pages += 1
        plt.close("all")

    plt.show = _show_and_savefig
    try:
        al.tears.create_full_tear_sheet(factor_data, long_short=True)
        _show_and_savefig()
    finally:
        plt.show = _original_show
        pdf.close()
        plt.close("all")

    print(f"Wrote {out_path} ({n_pages} pages)")
    return factor_data


In [10]:
# Review §4.1, then edit these (nothing auto-selected).
TEAR_MODE = "log_drawdown"          # ratio | log_drawdown
TEAR_WINDOW = 252            # paper default until you change after the screen
TEAR_NORMALIZE = False       # False → raw; True → cs

tear_col = near_52w_factor_col(TEAR_MODE, TEAR_WINDOW, TEAR_NORMALIZE)
print(f"Tear sheet factor: {tear_col}")
tear_data = run_full_tear(panel, tear_col, prices)


Tear sheet factor: near_52w_log_drawdown_252_raw


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Quantiles Statistics


,min,max,mean,std,count,count %
factor_quantile,,,,,,
1,-2.4918,-0.0466,-0.3946,0.2740,52860,20.1569
2,-0.8173,-0.0131,-0.1625,0.0930,52208,19.9083
3,-0.5756,-0.0045,-0.0929,0.0620,52108,19.8701
4,-0.4560,-0.0008,-0.0512,0.0430,52208,19.9083
5,-0.3298,0.0000,-0.0209,0.0261,52859,20.1565


Returns Analysis


...

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\performance.py:118: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  grouper.append(pd.Grouper(freq=by_time))
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:928: UserWarning: Skipping return periods that aren't exact multiples of days.
  warnings.warn(


Turnover Analysis


,1D,5D,21D
Quantile 1 Mean Turnover,0.0470,0.1020,0.2110
Quantile 2 Mean Turnover,0.1360,0.2850,0.5040
Quantile 3 Mean Turnover,0.2110,0.4150,0.6320
Quantile 4 Mean Turnover,0.2780,0.4850,0.6480
Quantile 5 Mean Turnover,0.1690,0.3270,0.5150


,1D,5D,21D
Mean Factor Rank Autocorrelation,0.9730,0.9090,0.7550


Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\02_research\notebooks\factor_tests\tearsheets\H-006_near_52w_log_drawdown_252_raw.pdf (3 pages)


## Conclusion

Fill after running the screen (edit this cell with your notes):

- **Best `ic_5d` combo:** mode / W / normalize = …
- **Normalize:** does `cs` or `raw` win more often in the §4.1 pivot? (and at 1d / 21d if materially different)
- **Paper default `W=252`:** hold up vs 63 / 126 / 504?
- **Keep / kill (H-006 alone):** …
- **Deferred:** compare survivors to H-001 / H-004 in a later pass if keeping.